# TP-3 — TinyGPT: ONNX export and an inference server

Author: Msc. Abraham R.

You will take the TinyGPT you fine-tuned in TP-2 and ship it: out of PyTorch, into a
portable computation graph, and behind an OpenAI-compatible API.

## From a training framework to a runtime

Nothing here trains. TP-1 pretrained a model and TP-2 taught it to follow instructions;
both ran inside a notebook, in PyTorch, one request at a time. Production does not work
that way, because a training framework is built for flexibility and a runtime is built
for one job:

| | PyTorch | an inference runtime |
|---|---|---|
| Built for | experimentation, autograd | one fixed graph, executed fast |
| Model is | Python code + weights | a serialized graph, no Python |
| Shipped as | a repo and an environment | one file |
| Runs on | where PyTorch installs | CPU, GPU, NPU, phone, browser |

This is the step behind runtimes like:
- ONNX Runtime (what you will use here)
- TensorRT, OpenVINO, CoreML
- llama.cpp / GGUF, which we walked through in class

**Part 1 — Export.** Turn the model into an ONNX graph and prove the graph computes what
PyTorch computed. Then export a second graph that carries the KV-cache.

**Part 2 — Quantize and measure.** Make it 4x smaller, then find out honestly what that
cost in quality and what it bought in speed.

**Part 3 — Serve.** Put the graph behind `POST /v1/chat/completions` and call it with the
official OpenAI client.

The model does not change in this notebook. It is the same checkpoint throughout — only
its representation changes.

## Tasks

**Part 1**
- Task I — export to ONNX, and write the parity test that proves it.
- Task II — export a second graph that takes and returns the **KV-cache**.
- Task III — port your TP-1 `generateV2` onto the runtime: temperature, top-k, top-p,
  and the cache.

**Part 2**
- Task IV — quantize to **INT8**, and measure what it cost in behaviour.
- Task V — benchmark PyTorch vs ONNX vs INT8, and split engine time from Python.

**Part 3**
- Task VI — implement `POST /v1/chat/completions`, OpenAI-compatible.
- Task VII — add **streaming**: the same endpoint, token by token over SSE.
- Task VIII — call your server with the official `openai` client, both ways.

## How to work on this

**Prerequisites:** TP-2. This notebook loads
`./checkpoints/tp2_full_ft/checkpoint_final.pt`. Without it everything still runs, on
random weights — the numbers stay meaningful, the generated text does not.

Tasks I, II, III, IV and VI each ship with a `check_*()` cell. Run it before moving on. Parity
bugs are silent: a wrong graph returns plausible numbers, and you will not notice until
the generations are subtly wrong.

**Versions matter here.** `torch.onnx.export` is an API in motion, and exporter behaviour
changes between releases. The versions this notebook was written against are printed by
the first cell; if yours differ and something breaks, that is the first thing to check.

References: [ONNX](https://onnx.ai/) ·
[ONNX Runtime](https://onnxruntime.ai/docs/) ·
[torch.onnx](https://pytorch.org/docs/stable/onnx.html) ·
[OpenAI chat completions](https://platform.openai.com/docs/api-reference/chat)

In [ ]:
import json
import os
import re
import threading
import time
import uuid
from collections import Counter
from typing import Dict, List, Literal, Optional, Sequence, Tuple

import numpy as np
import torch
from torch import nn
from transformers import AutoTokenizer

import onnx
import onnxruntime as ort
import requests
import uvicorn
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

from tinygpt import (
    GPTConfig,
    TinyGPT,
    describe,
    load_checkpoint,
    load_tinyshakespeare,
)

torch.manual_seed(1337)

# Everything here runs on CPU on purpose: it is the target this notebook can measure
# honestly, and the one every student has.
print(f"torch        {torch.__version__}")
print(f"onnx         {onnx.__version__}")
print(f"onnxruntime  {ort.__version__}")
print(f"providers    {ort.get_available_providers()}")

# Stage 0 — the model you are shipping

The TP-2 fine-tune, loaded exactly as TP-2 left it: same tokenizer, same four special
tokens, same untied embedding and head.

In [ ]:
USER, ASSISTANT, END, PAD = "<|user|>", "<|assistant|>", "<|end|>", "<|pad|>"
TP2_CKPT = "./checkpoints/tp2_full_ft/checkpoint_final.pt"
ONNX_DIR = "./onnx"
os.makedirs(ONNX_DIR, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.add_special_tokens({"additional_special_tokens": [USER, ASSISTANT, END, PAD]})
END_ID = tokenizer.convert_tokens_to_ids(END)
# The scaffolding a reply must never contain -- Task VI.
SCAFFOLD_IDS = [tokenizer.convert_tokens_to_ids(t) for t in (USER, ASSISTANT, PAD)]

# tie_weights=False for the same reason as in TP-2: the checkpoints keep token_emb and
# head as two separate matrices.
config = GPTConfig(vocab_size=len(tokenizer), tie_weights=False)
model = TinyGPT(config)

if os.path.exists(TP2_CKPT):
    load_checkpoint(model, TP2_CKPT, map_location="cpu")
    print(f"loaded fine-tuned weights from {TP2_CKPT}")
else:
    print("no TP-2 checkpoint found -- continuing on random weights.")
    print("(every measurement in this notebook still works; the text will be noise)")

model.eval()
describe(model, "TinyGPT (TP-2 fine-tuned)")
print(f"\nblock_size {config.block_size} | n_layer {config.n_layer} | "
      f"n_head {config.n_head} | vocab {config.vocab_size:,}")

## What we measure the graph against

Three things are needed throughout, all given: held-out text to score, a set of prompts,
and a decode loop.

**Accuracy is not one of them.** TP-2 measured `who_said` at roughly chance on 98
speakers, so an accuracy column here would read 0% for every artifact and tell you
nothing about what quantization did. Instead we use what does move:

- **cross-entropy** on held-out Shakespeare — the pretraining objective, the same number
  TP-2 used to measure forgetting. It is a real quality measure and it works no matter
  how bad the model is at any particular task;
- **agreement** with the fp32 model — how often two artifacts make the same choice.

Quality and fidelity are different questions, and after quantization you need both.

In [ ]:
# The same mining as TP-2, on the same corpus, with the same 1337 shuffle -- so these
# are examples the fine-tune did not train on.
_text = load_tinyshakespeare(None)
_speeches = re.findall(r"^([A-Z][A-Za-z' ]{1,30}):\n((?:.+\n)+)", _text, flags=re.M)
_counts = Counter(s for s, _ in _speeches)
_speeches = [(s, t) for s, t in _speeches if _counts[s] >= 20]
import random as _random
_random.Random(1337).shuffle(_speeches)
_held_out = _speeches[int(0.9 * len(_speeches)):]

BUDGET = config.block_size + 1


def _who_said(speaker: str, speech: str) -> Optional[Tuple[str, str]]:
    line = " ".join(speech.split())
    room = BUDGET - len(tokenizer.encode(f"{USER}who said: {ASSISTANT}")) \
                  - len(tokenizer.encode(speaker + END))
    if room < 5:
        return None
    body = tokenizer.decode(tokenizer.encode(line)[:room])
    return f"{USER}who said: {body}{ASSISTANT}", speaker


EVAL_SET = [p for p in (_who_said(s, t) for s, t in _held_out) if p is not None]

# Held-out text for the cross-entropy: the last 10% of the corpus, in block_size windows.
_ids = tokenizer(_text[-120_000:], add_special_tokens=False)["input_ids"]
LM_WINDOWS = [(np.array([_ids[i:i + config.block_size]], dtype=np.int64),
               np.array(_ids[i + 1:i + config.block_size + 1], dtype=np.int64))
              for i in range(0, 64 * config.block_size, config.block_size)]

# A few prompts to generate from, for reading rather than scoring.
DEMO_PROMPTS = [f"{USER}continue: To be, or not to be,{ASSISTANT}",
                f"{USER}who said: Now is the winter of our discontent{ASSISTANT}",
                f"{USER}reverse: sword{ASSISTANT}"]


def make_session(path: str, providers=("CPUExecutionProvider",)) -> ort.InferenceSession:
    """An ONNX Runtime session with the load-time chatter turned down."""
    so = ort.SessionOptions()
    so.log_severity_level = 3
    return ort.InferenceSession(path, so, providers=list(providers))


# Every session call in this notebook goes through here, so Task V can ask how much of
# the wall clock was the engine and how much was Python. Use it instead of sess.run.
ENGINE = {"seconds": 0.0}


def run_session(sess: ort.InferenceSession, feed: Dict[str, np.ndarray]) -> List[np.ndarray]:
    t0 = time.perf_counter()
    out = sess.run(None, feed)
    ENGINE["seconds"] += time.perf_counter() - t0
    return out


def _shape(value) -> str:
    t = value.type.tensor_type
    dims = [(d.dim_param or str(d.dim_value)) for d in t.shape.dim]
    return f"{onnx.TensorProto.DataType.Name(t.elem_type).lower()}[{','.join(dims)}]"


def show_graph(path: str, max_nodes: int = 12) -> None:
    """Signature, operator mix and the first few nodes of an ONNX file."""
    g = onnx.load(path).graph
    stored = {i.name for i in g.initializer}
    size = sum(len(i.raw_data) for i in g.initializer) / 1e6
    print(f"{path}  |  {len(g.node)} nodes  |  {len(g.initializer)} weight tensors "
          f"({size:.1f} MB)")
    print("  in   " + ", ".join(f"{i.name}: {_shape(i)}" for i in g.input))
    print("  out  " + ", ".join(f"{o.name}: {_shape(o)}" for o in g.output))
    print(f"  ops  {dict(Counter(n.op_type for n in g.node).most_common(8))}")
    for n in g.node[:max_nodes]:
        args = ", ".join(("*" + a if a in stored else a) for a in n.input)
        print(f"    {n.output[0][:28]:<30} = {n.op_type}({args[:52]})")
    if len(g.node) > max_nodes:
        print(f"    ... {len(g.node) - max_nodes} more    (* = a weight stored in the file)")


def greedy_torch(m: nn.Module, prompt: str, max_new_tokens: int = 14) -> str:
    """Reference decode loop, no cache, so ONNX and torch are compared like for like."""
    ids = tokenizer.encode(prompt)
    produced = []
    for _ in range(max_new_tokens):
        x = torch.tensor(ids[-config.block_size:], dtype=torch.long)[None, :]
        with torch.no_grad():
            nxt = int(m(x)[0, -1].argmax())
        if nxt == END_ID:
            break
        produced.append(nxt)
        ids.append(nxt)
    return tokenizer.decode(produced)


def greedy_onnx(sess: ort.InferenceSession, prompt: str, max_new_tokens: int = 14) -> str:
    """The same loop against a session whose only input is `input_ids`."""
    ids = tokenizer.encode(prompt)
    produced = []
    for _ in range(max_new_tokens):
        x = np.array([ids[-config.block_size:]], dtype=np.int64)
        nxt = int(run_session(sess, {"input_ids": x})[0][0, -1].argmax())
        if nxt == END_ID:
            break
        produced.append(nxt)
        ids.append(nxt)
    return tokenizer.decode(produced)


def cross_entropy(next_logits) -> float:
    """
    Mean cross-entropy over LM_WINDOWS, in nats per token.

    `next_logits(x) -> (T, vocab)` for a window `x` of shape (1, T).
    """
    total = 0.0
    for x, y in LM_WINDOWS:
        logits = next_logits(x).astype(np.float64)
        shift = logits - logits.max(axis=-1, keepdims=True)
        logZ = np.log(np.exp(shift).sum(axis=-1))
        total += float((logZ - shift[np.arange(len(y)), y]).mean())
    return total / len(LM_WINDOWS)


def torch_logits(x: np.ndarray) -> np.ndarray:
    with torch.no_grad():
        return model(torch.from_numpy(x))[0].numpy()


def onnx_logits(sess: ort.InferenceSession):
    return lambda x: sess.run(None, {"input_ids": x})[0][0]


print(f"{len(EVAL_SET):,} held-out who_said prompts | "
      f"{len(LM_WINDOWS)} held-out text windows")
print(f"torch cross-entropy: {cross_entropy(torch_logits):.4f} nats/token")

# Task I — export to ONNX

An ONNX file is a **computation graph**: the operators, their order, and the weights,
with no Python left. `torch.onnx.export` produces it by *tracing* — running your model
once on a dummy input and recording the operations that actually executed.

Tracing is why the export can be wrong in ways that do not raise. Anything decided by a
Python `if` on a tensor value is frozen as whatever happened during that one run.

**What to implement**

1. A wrapper `nn.Module`. `TinyGPT.forward` returns a tuple when `use_cache` or
   `return_weights` is set, and a bare tensor otherwise. The tracer needs exactly one
   path with one output, so wrap the model and return only `logits`.
2. `torch.onnx.export(...)` with:
   - `input_names=["input_ids"]`, `output_names=["logits"]`;
   - `dynamic_axes` marking **batch** and **sequence** as dynamic on both. Without this
     the graph is frozen at the dummy shape and only accepts prompts of that exact
     length;
   - `opset_version=17`, `do_constant_folding=True`.

**Two traps, both silent**

- **`export()` leaves the module in training mode.** Dropout is then live on the PyTorch
  side, your parity check reads ~4 instead of ~1e-06, and you spend the evening blaming
  the exporter. Call `.eval()` again after exporting — the self-check asserts this.
- **Parity is the only proof you have.** A traced graph that runs is not a graph that is
  correct. Compare logits against PyTorch at *several* sequence lengths, not one.

In [ ]:
class ExportWrapper(nn.Module):
    """One input, one output, one path through the model -- what the tracer needs."""

    def __init__(self, m: nn.Module):
        super().__init__()
        # TODO: Task I
        raise NotImplementedError

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        # TODO: Task I -- return logits only
        raise NotImplementedError


def export_onnx(m: nn.Module, path: str) -> str:
    """
    Trace `m` to an ONNX graph at `path` with dynamic batch and sequence axes.

    Returns `path`. Remember what tracing does to `m.training`.
    """
    # TODO: Task I
    raise NotImplementedError

## Self-check

In [ ]:
def check_export():
    path = export_onnx(model, f"{ONNX_DIR}/tinygpt.onnx")
    assert os.path.exists(path), "nothing was written"

    assert not model.training, \
        "the model is in train mode -- export() put it there; call .eval() again"

    sess = make_session(path)
    names = [i.name for i in sess.get_inputs()]
    assert names == ["input_ids"], f"inputs are {names}"

    worst = 0.0
    for T in (1, 8, 17, config.block_size):
        x = torch.randint(0, config.vocab_size, (1, T), dtype=torch.long)
        with torch.no_grad():
            ref = model(x).numpy()
        got = sess.run(None, {"input_ids": x.numpy()})[0]
        assert got.shape == ref.shape, f"T={T}: {got.shape} != {ref.shape}"
        worst = max(worst, float(np.abs(ref - got).max()))
    assert worst < 1e-4, f"logits disagree by {worst:.2e} -- the graph is not the model"

    # A dynamic batch axis is what lets a server batch requests later.
    x = torch.randint(0, config.vocab_size, (4, 8), dtype=torch.long)
    assert sess.run(None, {"input_ids": x.numpy()})[0].shape[0] == 4, \
        "batch axis is frozen -- check dynamic_axes"

    print(f"parity     max |torch - onnx| = {worst:.2e}  (seq 1/8/17/{config.block_size})")
    print(f"sample     {greedy_onnx(sess, DEMO_PROMPTS[1])!r}")
    print("all checks passed\n")
    # The same reader as the hand-made graph -- now pointed at a transformer.
    show_graph(path, max_nodes=6)
    return sess


fp32_sess = check_export()

## Look at what you just made

`check_export` printed the graph's signature and operator mix. Now open the file itself.

[**Netron**](https://netron.app) is the standard viewer for this format, and the cell
below embeds it in the notebook if you have it (`pip install netron`). Otherwise open
[netron.app](https://netron.app) and drag `onnx/tinygpt.onnx` onto the page — it renders
locally in your browser, the file is not uploaded anywhere.

Things worth finding, because they are the point of the exercise:

- **`Gather` at the start and a 50,261-wide `MatMul` at the end** — the embedding and the
  output head, the two matrices TP-2 measured at 98% of this model.
- **Your `block_size` attention mask, baked in as a constant.** It is a `Slice` over a
  stored `tril`, not code.
- **No Python, no classes, no `forward`.** `TinyGPT` is gone; what remains is arithmetic.
- **Two identical-looking blocks.** Your `n_layer = 2`, unrolled — the loop that wrote
  them does not exist in the file.

Find one operator you did not write and explain where it came from. Tracing adds
`Shape`, `Unsqueeze` and `Constant` nodes for every Python-level shape computation your
`forward` performed, which is why 221 nodes come out of a model you could read in a
minute.

In [ ]:
def view_graph(path: str, height: int = 600):
    """Embed Netron if it is installed; otherwise say how to view the file by hand."""
    try:
        import netron
    except ImportError:
        print(f"netron is not installed. Either:\n"
              f"  pip install netron      (then re-run this cell)\n"
              f"  or open https://netron.app and drag in {path}")
        return show_graph(path, max_nodes=12)

    from IPython.display import IFrame
    host, port = netron.start(path, address=("localhost", 0), browse=False)
    print(f"serving {path} at http://{host}:{port}   (netron.stop() to close)")
    return IFrame(f"http://{host}:{port}", width="100%", height=height)


view_graph(f"{ONNX_DIR}/tinygpt.onnx")

# Task II — export the KV-cache

The graph you just exported re-reads the whole prompt for every token. TP-1 Task III
measured what a KV-cache saves; now it has to survive the trip into ONNX.

A runtime has no Python objects to hold state between calls, so the cache cannot live in
a list on the side. **It becomes graph input and graph output**: the caller passes the
past in and gets the present back. This is exactly how exported GPT-2, Llama and every
other decoder is served.

`TinyGPT` keeps one stacked tensor per layer, shaped
`(2, batch, n_head, past, head_dim)` — key and value stacked on axis 0. With
`n_layer = 2` that is 2 tensors in and 2 tensors out, plus `input_ids`.

**What to implement**

1. `KVWrapper`, whose `forward(input_ids, *past)` calls the model with
   `kv_cache=list(past), use_cache=True` and returns `(logits, *present)`.
2. `export_onnx_kv`, tracing it with a real cache from a real forward pass, naming the
   tensors `past_0..` and `present_0..`, and marking the **past-length axis as dynamic**
   on every one of them.

**The trap here is the mask.** Attention slices `tril[past:T_k, :T_k]`, where `past`
comes from a tensor shape. If that slice is traced as a constant, the graph works
perfectly at the length you traced and is wrong everywhere else — so the self-check
runs several past lengths, not one.

**Hint.** Build the cache to trace with:

```python
with torch.no_grad():
    _, cache = m(some_ids, use_cache=True)
```

In [ ]:
class KVWrapper(nn.Module):
    """Cache in, cache out -- a runtime keeps no state between calls."""

    def __init__(self, m: nn.Module):
        super().__init__()
        # TODO: Task II
        raise NotImplementedError

    def forward(self, input_ids: torch.Tensor, *past: torch.Tensor):
        # TODO: Task II -- return (logits, *present)
        raise NotImplementedError


def export_onnx_kv(m: nn.Module, path: str) -> str:
    """
    Export a one-step graph that takes the past cache and returns the present one.

    Inputs:  input_ids, past_0 .. past_{n_layer-1}
    Outputs: logits,    present_0 .. present_{n_layer-1}
    """
    # TODO: Task II
    raise NotImplementedError

## Self-check

In [ ]:
def empty_cache(batch: int = 1) -> List[np.ndarray]:
    """What you feed at the first step: a cache with zero past positions."""
    head_dim = config.n_embd // config.n_head
    return [np.zeros((2, batch, config.n_head, 0, head_dim), dtype=np.float32)
            for _ in range(config.n_layer)]


def check_kv_export():
    path = export_onnx_kv(model, f"{ONNX_DIR}/tinygpt.kv.onnx")
    assert not model.training, "export() left the model in train mode again"

    sess = make_session(path)
    ins = [i.name for i in sess.get_inputs()]
    outs = [o.name for o in sess.get_outputs()]
    assert ins == ["input_ids"] + [f"past_{i}" for i in range(config.n_layer)], ins
    assert outs == ["logits"] + [f"present_{i}" for i in range(config.n_layer)], outs

    # The point of the test: past lengths the export never saw.
    worst = 0.0
    for past_len in (0, 3, 8, 20, config.block_size - 1):
        ids = torch.randint(0, config.vocab_size, (1, max(past_len, 1)), dtype=torch.long)
        if past_len == 0:
            cache = empty_cache()
        else:
            with torch.no_grad():
                _, c = model(ids, use_cache=True)
            cache = [t.numpy() for t in c]

        step = torch.randint(0, config.vocab_size, (1, 1), dtype=torch.long)
        feed = {"input_ids": step.numpy()}
        feed.update({f"past_{i}": c for i, c in enumerate(cache)})
        got = sess.run(None, feed)

        torch_cache = None if past_len == 0 else [torch.from_numpy(c) for c in cache]
        with torch.no_grad():
            ref, present = model(step, kv_cache=torch_cache, use_cache=True)
        worst = max(worst, float(np.abs(ref.numpy() - got[0]).max()))
        assert got[1].shape[3] == past_len + 1, \
            f"present cache is {got[1].shape[3]} long, expected {past_len + 1}"
        assert np.abs(present[0].numpy() - got[1]).max() < 1e-4, "returned cache differs"

    assert worst < 1e-4, (f"logits disagree by {worst:.2e} -- the mask was probably "
                          f"traced as a constant")
    print(f"parity     max |torch - onnx| = {worst:.2e}  (past 0/3/8/20/{config.block_size - 1})")
    print("all checks passed\n")
    # Read the signature: the cache is now part of the contract, in and out.
    show_graph(path, max_nodes=0)
    return sess


kv_sess = check_kv_export()

# Task III — `generateV2`, ported to the runtime

TP-1 Task I had you write `generateV2`: greedy, temperature, top-k, top-p, with an
optional KV-cache. That was PyTorch. **Decoding is not part of the model** — it never
was, which is why none of it appears in the graph you exported. It is a loop the caller
writes, around whatever computes logits.

So port it. Same knobs, same semantics as TP-1, against an ONNX session instead of an
`nn.Module`. Two differences, and they are the lesson:

- **There is no `model.forward`, only a feed dict.** Truncation, cache handling and
  position bookkeeping are all yours now.
- **The cache is not hidden state.** With `use_cache=True` you drive the Task II graph:
  feed `past_*`, read `present_*`, and hand them back on the next step.

**What to implement**

1. `sample_next(logits, ...)` — the decoding rules, as a pure function on one row of
   logits. Greedy when `do_sample=False`; otherwise temperature, then top-k, then top-p,
   then sample. `suppress` lists token ids that must never be chosen (Part 3 needs it for
   the chat scaffolding; TP-2 Task V hit the same problem).
2. `stream_tokens(sess, prompt, ...)` — a **generator** yielding token ids until `<|end|>`
   or `max_new_tokens`. `use_cache=False` drives the Task I graph, `use_cache=True` the
   Task II one.

A generator, not a string, because Part 3 needs the tokens as they appear. `generate_onnx`
below wraps it back into text and is given.

**Details that matter**

- Order the knobs as TP-1 did: temperature scales the logits, then top-k truncates, then
  top-p truncates, then you sample. Applying top-p before temperature gives a different
  distribution.
- `top_p` keeps the **smallest set whose cumulative probability reaches p**, so
  `top_p` near 0 is effectively greedy, and `top_p=1.0` is no truncation at all.
- With the cache, only the **new** token goes in as `input_ids` after the first call, and
  the cache must be trimmed at `block_size` — the graph asserts it.
- Call `run_session(sess, feed)` rather than `sess.run(...)`, so Task V can measure how
  much of the time was actually the engine.
- Seed with `np.random.default_rng(seed)` so a sampled run can be reproduced.

In [ ]:
def sample_next(logits: np.ndarray, do_sample: bool = True,
                temperature: float = 1.0, top_k: Optional[int] = None,
                top_p: Optional[float] = None, suppress: Sequence[int] = (),
                rng: Optional[np.random.Generator] = None) -> int:
    """
    One row of logits -> one token id.

    do_sample=False -> arg-max, and every other knob is ignored (as in TP-1).
    Order: temperature, then top-k, then top-p, then sample.
    """
    # TODO: Task III
    raise NotImplementedError


def stream_tokens(sess: ort.InferenceSession, prompt: str, max_new_tokens: int = 24,
                  use_cache: bool = True, do_sample: bool = True,
                  temperature: float = 1.0, top_k: Optional[int] = None,
                  top_p: Optional[float] = None, suppress: Sequence[int] = (),
                  seed: Optional[int] = None):
    """
    Yields generated token ids, stopping at END_ID or after max_new_tokens.

    use_cache=False -> `sess` is the Task I graph (input_ids only).
    use_cache=True  -> `sess` is the Task II graph (past_* in, present_* out).
    """
    # TODO: Task III
    raise NotImplementedError


def generate_onnx(sess: ort.InferenceSession, prompt: str, **kw) -> str:
    """Given: the text-returning wrapper, so every caller agrees on the contract."""
    return tokenizer.decode(list(stream_tokens(sess, prompt, **kw)))

## Self-check

In [ ]:
def check_generate():
    prompt = DEMO_PROMPTS[1]
    greedy = dict(do_sample=False, max_new_tokens=14)

    # 1. the cacheless path must reproduce the reference loop exactly
    mine = generate_onnx(fp32_sess, prompt, use_cache=False, **greedy)
    assert mine == greedy_onnx(fp32_sess, prompt), f"{mine!r} != reference"

    # 2. and the cached path must agree with it -- same weights, same choices
    cached = generate_onnx(kv_sess, prompt, use_cache=True, **greedy)
    assert cached == mine, f"cache changes the output: {cached!r} != {mine!r}"

    # 3. do_sample=False ignores every other knob
    loud = generate_onnx(kv_sess, prompt, temperature=5.0, top_k=999, top_p=0.3, **greedy)
    assert loud == mine, "do_sample=False must ignore temperature/top_k/top_p"

    # 4. the degenerate settings of each knob collapse to greedy
    assert generate_onnx(kv_sess, prompt, max_new_tokens=14, top_k=1, seed=0) == mine, \
        "top_k=1 must be greedy"
    assert generate_onnx(kv_sess, prompt, max_new_tokens=14, top_p=1e-6, seed=0) == mine, \
        "top_p -> 0 keeps only the most likely token"

    # 5. sampling actually samples, and a seed makes it reproducible
    draws = {generate_onnx(kv_sess, prompt, max_new_tokens=14, temperature=2.0, seed=s)
             for s in range(8)}
    assert len(draws) > 1, "temperature=2.0 produced one single output -- not sampling"
    a = generate_onnx(kv_sess, prompt, max_new_tokens=14, temperature=2.0, seed=7)
    b = generate_onnx(kv_sess, prompt, max_new_tokens=14, temperature=2.0, seed=7)
    assert a == b, "same seed must give the same text"

    # 6. suppressed ids are never emitted, however hot the sampling
    banned = [int(np.array(tokenizer.encode(mine))[0])] if mine else []
    if banned:
        for seed in range(4):
            ids = list(stream_tokens(kv_sess, prompt, max_new_tokens=14, temperature=2.0,
                                     suppress=banned, seed=seed))
            assert banned[0] not in ids, "a suppressed id was sampled"

    # 7. sample_next is a pure function -- test it without a model at all
    logits = np.array([0.0, 10.0, 5.0, -1.0])
    assert sample_next(logits, do_sample=False) == 1
    assert sample_next(logits, do_sample=False, suppress=[1]) == 2
    rng = np.random.default_rng(0)
    assert sample_next(logits, top_k=1, rng=rng) == 1

    print(f"greedy       {mine!r}")
    print(f"cached       {cached!r}  (identical)")
    print(f"temp 2.0     {sorted(draws)[0]!r}")
    print(f"distinct outputs over 8 seeds: {len(draws)}")
    print("all checks passed")


check_generate()

# Task IV — INT8, and what it costs

Dynamic quantization stores the weights as 8-bit integers and dequantizes them on the
fly. It needs no calibration data and is one function call:

```python
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(fp32_path, int8_path, weight_type=QuantType.QUInt8)
```

**The test from Task I no longer applies.** Quantization changes the arithmetic on
purpose. Logit parity against PyTorch will be large — around 1e-01 rather than 1e-06 —
and that is not a bug to fix. It means the question has changed: not *is this the same
computation?* but **is this the same model, behaviourally?**

So measure behaviour, on two axes:

- **quality** — cross-entropy on held-out text, fp32 against INT8. Did the model get
  worse at modelling language?
- **fidelity** — agreement with fp32. **Top-1 agreement** is how often both pick the
  same next token; **reply agreement** is how often a whole greedy decode comes out
  character for character identical. The second is much harsher, because one different
  token early changes everything after it.

Report size, parity, cross-entropy and both agreements together. One of those numbers
will look alarming and the rest will not; your write-up should say which one you trust
and why.

The self-check below verifies the *measurement* rather than the model: a comparison that
cannot recognise a session as identical to itself would make every number here noise.

In [ ]:
def quantize_int8(fp32_path: str, int8_path: str) -> str:
    """Dynamic INT8 quantization of an existing ONNX file. Returns int8_path."""
    # TODO: Task IV
    raise NotImplementedError


def top1_agreement(sess_a: ort.InferenceSession, sess_b: ort.InferenceSession,
                   n: int = 60) -> float:
    """
    Share of prompts where both sessions predict the same next token.

    Both sessions take `input_ids` only. One forward pass each, no decoding.
    """
    # TODO: Task IV
    raise NotImplementedError


def reply_agreement(sess_a: ort.InferenceSession, sess_b: ort.InferenceSession,
                    n: int = 40) -> float:
    """Share of prompts where the two greedy decodes are identical strings."""
    # TODO: Task IV
    raise NotImplementedError


int8_path = quantize_int8(f"{ONNX_DIR}/tinygpt.onnx", f"{ONNX_DIR}/tinygpt.int8.onnx")
int8_sess = make_session(int8_path)

# TODO: Task IV -- fill this table in
#   size MB | max |torch - onnx| | cross-entropy | top-1 agreement | reply agreement
raise NotImplementedError

## Self-check

In [ ]:
def check_quantized():
    # A comparison function that cannot see itself as identical is broken, and every
    # number in the table above would then be noise.
    assert top1_agreement(fp32_sess, fp32_sess) == 1.0, \
        "a session must agree with itself 100% of the time"
    assert reply_agreement(fp32_sess, fp32_sess) == 1.0, \
        "a session must reproduce its own replies exactly"

    top1 = top1_agreement(fp32_sess, int8_sess)
    replies = reply_agreement(fp32_sess, int8_sess)
    assert 0.0 <= top1 <= 1.0 and 0.0 <= replies <= 1.0, "agreement is a share, not a count"
    assert replies <= top1 + 1e-9, \
        "whole replies cannot agree more often than single tokens do"

    # INT8 perturbs the weights; it does not replace the model. A very low number here
    # is a bug in the comparison or the wrong session, not quantization damage.
    assert top1 > 0.5, f"top-1 agreement {top1:.1%} is too low to be quantization alone"
    drift = cross_entropy(onnx_logits(int8_sess)) - cross_entropy(onnx_logits(fp32_sess))
    assert abs(drift) < 0.5, f"cross-entropy moved {drift:+.3f} nats -- that is not rounding"

    assert os.path.getsize(int8_path) < 0.6 * os.path.getsize(f"{ONNX_DIR}/tinygpt.onnx"), \
        "the int8 file is not meaningfully smaller -- did quantization run?"

    print(f"self-agreement   100.0%  (the measurement works)")
    print(f"fp32 vs int8     top-1 {top1:.1%} | replies {replies:.1%} | "
          f"cross-entropy {drift:+.4f}")
    print("all checks passed")


check_quantized()

# Task V — benchmark

Four ways to produce the same tokens: PyTorch, ONNX without the cache, ONNX with the
cache, and INT8. Your `stream_tokens` already drives three of them — run it greedy, so
the comparison is about the runtime and not about sampling luck.

Measure two things per run, not one:

- **wall time** for the whole loop;
- **engine time** — `ENGINE["seconds"]`, which `run_session` has been accumulating for
  you. Reset it before each run and read it after.

The gap between them is Python: tokenizing, slicing, building the feed dict, sampling.
At 6.5M parameters that gap is not a rounding error, and noticing it is the point of the
exercise. It is also why real servers are not written as a Python `for` loop.

**Rules for a benchmark worth reporting:** warm up first (the first `run` builds
kernels), use the same prompt and the same token count everywhere, and repeat enough to
see the noise. State what machine it ran on.

In [ ]:
N_TOKENS, WARMUP, REPEATS = 32, 3, 3


def timed(decode, n_tokens: int = N_TOKENS) -> Tuple[float, float]:
    """
    Warm up, then run `decode(n_tokens)` and return (wall seconds, engine seconds).

    Engine time comes from ENGINE["seconds"] -- reset it right before timing.
    Take the best of REPEATS runs: the minimum is the least noisy estimate.
    """
    # TODO: Task V
    raise NotImplementedError


def torch_decode(n_tokens: int) -> None:
    """Given: the PyTorch baseline, greedy, no cache -- the thing ONNX has to beat."""
    ids = tokenizer.encode(DEMO_PROMPTS[0])
    for _ in range(n_tokens):
        x = torch.tensor(ids[-config.block_size:], dtype=torch.long)[None, :]
        t0 = time.perf_counter()
        with torch.no_grad():
            logits = model(x)
        ENGINE["seconds"] += time.perf_counter() - t0
        ids.append(int(logits[0, -1].argmax()))


# TODO: Task V -- time these four and print ms/token, tok/s, the engine/python split
# and the speedup over torch:
#   torch cpu | onnx fp32 (no cache) | onnx fp32 (kv cache) | onnx int8 (no cache)
#
# For the ONNX rows call your own generator, greedy and without stopping early:
#   list(stream_tokens(sess, DEMO_PROMPTS[0], max_new_tokens=n, use_cache=...,
#                      do_sample=False))
raise NotImplementedError

# Questions

Answer these from the numbers you just produced. No re-running anything.

1. Your INT8 model is about **4x smaller** but nowhere near 4x faster. Explain both
   numbers with the same fact about this architecture — TP-2 measured it, and it is the
   reason "parameter-efficient" was a real question there.
2. The KV-cache graph does strictly less arithmetic per token than the cacheless one.
   Did it win by the margin you expected? Whatever the answer, explain it in terms of
   `block_size = 32` and the Python share in your own table.
3. Logit parity is the right test in Task I and the wrong one in Task IV. State what
   each of your three tests — parity, cross-entropy, agreement — actually measures, and
   give a change to a model that agreement would catch while cross-entropy stayed flat.
4. Tracing records one execution. Name a change to `TinyGPT.forward` that would export
   cleanly, pass a single-length parity check, and still be wrong in production — and
   say which line of the Task II self-check would catch it.
5. You exported the same weights three times. Which artifact would you actually ship to
   a phone, and which to a server with a GPU? Defend both choices with a number.

---

-
-
-
-
-

# Task VI — an OpenAI-compatible endpoint

The graph runs. Now give it the interface the rest of the world already speaks.

Every hosted model — OpenAI, and llama.cpp, vLLM, Ollama and TGI in their compatibility
modes — answers `POST /v1/chat/completions` with the same JSON. Speak that shape and
every existing client works against your model unchanged. That is the entire lesson of
this part: **the API contract is the product boundary, not the model.**

The request you must accept and the response you must return:

```json
{ "model": "tinygpt", "messages": [{"role": "user", "content": "continue: To be,"}],
  "max_tokens": 24, "temperature": 0.8 }
```

```json
{ "id": "chatcmpl-...", "object": "chat.completion", "created": 1700000000,
  "model": "tinygpt",
  "choices": [{ "index": 0,
                "message": {"role": "assistant", "content": "..."},
                "finish_reason": "stop" }],
  "usage": {"prompt_tokens": 12, "completion_tokens": 7, "total_tokens": 19} }
```

**What to implement**

1. `render_prompt(messages)` — OpenAI messages into TP-2's template. A `system` message
   has nowhere to go in a template that only knows `<|user|>` and `<|assistant|>`:
   decide what you do with it and say so in your write-up.
2. `decode_args(req)` — translate the request's knobs into the keyword arguments your
   Task III `stream_tokens` takes. `temperature: 0` means greedy, and `SCAFFOLD_IDS`
   must be suppressed.
3. `generate_reply(req)` — run the decode. Return the text **and** why you stopped, plus
   the token counts.
4. `build_response(req)` — assemble the JSON above.

No new decoding code: Task III already wrote it. This task is the translation layer
between a public API and your generator, which is most of what a serving stack is.

The route itself is registered a few cells down, once both this task and Task VII are
written. Registering the same path twice in FastAPI does not replace the handler — the
first one wins, silently — so all the routing lives in one given cell.

**Three details that are the actual grade**

- **`finish_reason`** is `"stop"` when the model emitted `<|end|>` and `"length"` when it
  hit `max_tokens`. Clients branch on this — it is how a caller knows whether to ask for
  more.
- **`usage`** must count real tokens: prompt, completion, and their sum.
- **Suppress the scaffolding.** `SCAFFOLD_IDS` holds `<|user|>`, `<|assistant|>` and
  `<|pad|>`. Set those logits to `-inf` before sampling, or they appear *inside* replies —
  the same bug TP-2 Task V warned about, now shipping to users.

In [ ]:
class Message(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str


class ChatRequest(BaseModel):
    model: str = "tinygpt"
    messages: List[Message]
    max_tokens: int = 24
    temperature: float = 0.8
    top_p: float = 1.0            # part of the OpenAI contract -- Task III implements it
    stream: bool = False          # Task VII


app = FastAPI(title="TinyGPT")


def render_prompt(messages: List[Message]) -> str:
    """OpenAI messages -> the TP-2 chat template, ending at <|assistant|>."""
    # TODO: Task VI
    raise NotImplementedError


def decode_args(req: ChatRequest) -> Dict:
    """
    The keyword arguments your Task III `stream_tokens` needs for this request.

    temperature 0 means greedy; the scaffolding must never be sampled.
    """
    # TODO: Task VI
    raise NotImplementedError


def generate_reply(req: ChatRequest) -> Tuple[str, str, int, int]:
    """Returns (text, finish_reason, n_prompt_tokens, n_completion_tokens)."""
    # TODO: Task VI -- decode with stream_tokens; fewer tokens than max_tokens
    #                 means it stopped on <|end|>
    raise NotImplementedError


def build_response(req: ChatRequest) -> Dict:
    """The non-streaming response documented above."""
    # TODO: Task VI
    raise NotImplementedError

# Task VII — streaming

Your endpoint makes the caller wait for the last token before it sees the first. At 24
tokens on a CPU that is a pause; at 500 tokens it is a dead UI. Every chat product you
have used streams instead, and the mechanism is older and simpler than the model:
**server-sent events**, a long-lived HTTP response written one chunk at a time.

The wire format is plain text. Each event is a `data:` line with one JSON object,
separated by a blank line, and the stream ends with a literal sentinel:

```
data: {"id":"chatcmpl-abc","object":"chat.completion.chunk","created":1700000000,"model":"tinygpt","choices":[{"index":0,"delta":{"role":"assistant"},"finish_reason":null}]}

data: {"id":"chatcmpl-abc", ... ,"choices":[{"index":0,"delta":{"content":"DUKE"},"finish_reason":null}]}

data: {"id":"chatcmpl-abc", ... ,"choices":[{"index":0,"delta":{},"finish_reason":"stop"}]}

data: [DONE]

```

Three differences from Task VI, all of which clients depend on:

- `object` is **`chat.completion.chunk`**, and each choice carries a **`delta`**, not a
  `message`. The first delta announces the role, the middle ones carry text, the last is
  empty and carries `finish_reason`.
- The `id` is the **same for every chunk** of one response.
- `usage` is absent. It is not known until the end, and OpenAI only sends it when asked
  via `stream_options`. Leaving it out is the compatible default.

**What to implement**

`sse_events(req)`, a generator yielding those `data: ...\n\n` strings. Decode by
iterating your Task III `stream_tokens` — one decode loop, two presentations.

**The trap: do not decode tokens one at a time.** Byte-level BPE splits characters across
tokens, so `tokenizer.decode([one_id])` can hand you half a character. Decode everything
produced so far and emit only the part you have not sent yet:

```python
text = tokenizer.decode(produced)
delta, sent = text[len(sent):], text
```

In [ ]:
def sse_chunk(chunk_id: str, model: str, delta: Dict,
              finish_reason: Optional[str]) -> str:
    """One `data: {...}` event, blank line included. Given -- the shape is the contract."""
    payload = {
        "id": chunk_id,
        "object": "chat.completion.chunk",
        "created": int(time.time()),
        "model": model,
        "choices": [{"index": 0, "delta": delta, "finish_reason": finish_reason}],
    }
    return f"data: {json.dumps(payload)}\n\n"


def sse_events(req: ChatRequest):
    """
    Yields the event stream for one request.

    Order: a role delta, then content deltas, then an empty delta carrying
    finish_reason, then `data: [DONE]`.
    """
    # TODO: Task VII
    raise NotImplementedError

## Start it

The route is registered here, once, and dispatches on `req.stream` to the two things you
wrote. `uvicorn` then runs in a background thread so the notebook stays usable, bound to
`127.0.0.1` so nothing outside this machine can reach it.

Both self-checks call the endpoint over real HTTP and read the contract back.

In [ ]:
@app.post("/v1/chat/completions")
def chat_completions(req: ChatRequest):
    """The only route. Task VI answers in one piece, Task VII answers in many."""
    if req.stream:
        return StreamingResponse(sse_events(req), media_type="text/event-stream")
    return build_response(req)


PORT = 8000
_config = uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="error")
server = uvicorn.Server(_config)
threading.Thread(target=server.run, daemon=True).start()

for _ in range(200):                      # it binds the socket in well under a second
    if server.started:
        break
    time.sleep(0.05)
print(f"serving on http://127.0.0.1:{PORT}  (started={server.started})")


def check_api():
    url = f"http://127.0.0.1:{PORT}/v1/chat/completions"
    body = {"model": "tinygpt", "temperature": 0.0, "max_tokens": 16,
            "messages": [{"role": "user", "content": "who said: Now is the winter"}]}
    r = requests.post(url, json=body, timeout=60)
    assert r.status_code == 200, f"{r.status_code}: {r.text[:200]}"
    data = r.json()

    for key in ("id", "object", "created", "model", "choices", "usage"):
        assert key in data, f"response is missing {key!r}"
    assert data["object"] == "chat.completion", data["object"]
    choice = data["choices"][0]
    assert choice["message"]["role"] == "assistant", choice["message"]
    assert choice["finish_reason"] in ("stop", "length"), choice["finish_reason"]

    usage = data["usage"]
    assert usage["prompt_tokens"] + usage["completion_tokens"] == usage["total_tokens"], usage
    assert usage["completion_tokens"] > 0, "nothing was generated"

    content = choice["message"]["content"]
    for tok in (USER, ASSISTANT, PAD):
        assert tok not in content, f"{tok} leaked into the reply -- suppress SCAFFOLD_IDS"

    # max_tokens is a promise, not a suggestion.
    short = requests.post(url, json={**body, "max_tokens": 2}, timeout=60).json()
    assert short["usage"]["completion_tokens"] <= 2, short["usage"]
    if short["choices"][0]["finish_reason"] == "length":
        assert short["usage"]["completion_tokens"] == 2, short["usage"]

    # temperature must actually reach the sampler. Greedy twice is identical; hot
    # sampling over several draws must not be.
    greedy = {requests.post(url, json={**body, "temperature": 0.0}, timeout=60)
              .json()["choices"][0]["message"]["content"] for _ in range(2)}
    assert len(greedy) == 1, "temperature=0 must be deterministic"
    hot = {requests.post(url, json={**body, "temperature": 2.0}, timeout=60)
           .json()["choices"][0]["message"]["content"] for _ in range(6)}
    assert len(hot) > 1, "temperature=2.0 gave one single reply -- is it reaching sample_next?"

    # A second turn has to reach the model: a chat API that forgets the conversation
    # is broken, not opinionated.
    multi = requests.post(url, json={**body, "messages": [
        {"role": "user", "content": "who said: Now is the winter"},
        {"role": "assistant", "content": "GLOUCESTER"},
        {"role": "user", "content": "continue: To be, or not"}]}, timeout=60).json()
    assert multi["usage"]["prompt_tokens"] > usage["prompt_tokens"], \
        "a multi-turn conversation produced no more prompt tokens than one message"

    # The system role is a design decision, not a rule -- so this reports, not asserts.
    sys_usage = requests.post(url, json={**body, "messages": [
        {"role": "system", "content": "You are a Shakespearean actor."},
        {"role": "user", "content": "who said: Now is the winter"}]},
        timeout=60).json()["usage"]
    reached = sys_usage["prompt_tokens"] > usage["prompt_tokens"]

    print(f"content      {content!r}")
    print(f"finish       {choice['finish_reason']}")
    print(f"usage        {usage['prompt_tokens']}+{usage['completion_tokens']}"
          f"={usage['total_tokens']}")
    print(f"system role  {'reaches the model' if reached else 'is dropped'}"
          f"  ({usage['prompt_tokens']} -> {sys_usage['prompt_tokens']} prompt tokens)"
          f"  -- defend your choice in the questions")
    print("all checks passed")


def check_stream():
    url = f"http://127.0.0.1:{PORT}/v1/chat/completions"
    body = {"model": "tinygpt", "temperature": 0.0, "max_tokens": 16, "stream": True,
            "messages": [{"role": "user", "content": "continue: To be, or not to be,"}]}
    r = requests.post(url, json=body, stream=True, timeout=60)
    assert r.status_code == 200, f"{r.status_code}: {r.text[:200]}"
    assert "text/event-stream" in r.headers.get("content-type", ""), r.headers

    events = [l[len("data: "):] for l in r.iter_lines(decode_unicode=True)
              if l and l.startswith("data: ")]
    assert events, "no data: events"
    assert events[-1] == "[DONE]", f"stream ends with {events[-1]!r}, not [DONE]"

    chunks = [json.loads(e) for e in events[:-1]]
    assert all(c["object"] == "chat.completion.chunk" for c in chunks), \
        "object must be chat.completion.chunk while streaming"
    assert len({c["id"] for c in chunks}) == 1, "every chunk needs the same id"
    assert chunks[0]["choices"][0]["delta"].get("role") == "assistant", \
        "the first delta announces the role"
    assert all(c["choices"][0]["finish_reason"] is None for c in chunks[:-1]), \
        "only the last chunk carries finish_reason"
    assert chunks[-1]["choices"][0]["finish_reason"] in ("stop", "length"), chunks[-1]

    text = "".join(c["choices"][0]["delta"].get("content", "") for c in chunks)
    assert text, "no content was streamed"
    for tok in (USER, ASSISTANT, PAD):
        assert tok not in text, f"{tok} leaked into the stream"
    assert "�" not in text, \
        "replacement char -- you are decoding tokens one at a time"

    print(f"chunks       {len(chunks)} + [DONE]")
    print(f"streamed     {text!r}")
    print(f"finish       {chunks[-1]['choices'][0]['finish_reason']}")
    print("all checks passed")


check_api()
print()
check_stream()

# Task VIII — call it with the OpenAI client

The self-check spoke raw HTTP. This is the part worth the trouble: the **official
`openai` client**, the same library that talks to GPT-5, pointed at your laptop.

Nothing about the client is modified. It sends its usual request, parses the response
into its usual `ChatCompletion` object, and never learns that behind the URL sits a
6.5M-parameter Shakespeare model running on a CPU.

Do it both ways:

- **buffered** — a few messages, including one that stops on `<|end|>` and one that runs
  out of `max_tokens`, reading both `finish_reason` values back;
- **streaming** — `stream=True`, printing each `chunk.choices[0].delta.content` as it
  arrives. The client hands you `ChatCompletionChunk` objects parsed from the SSE you
  wrote in Task VII.

If the streamed text and the buffered text can differ for the same prompt, say why in
your write-up.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=f"http://127.0.0.1:{PORT}/v1", api_key="not-needed")

# TODO: Task VIII
#   a) send at least three messages and print reply, finish_reason and usage for each;
#      include one that stops on <|end|> and one that runs out of max_tokens.
#   b) send one with stream=True and print the deltas as they arrive.
raise NotImplementedError

# Questions

1. `finish_reason` distinguishes `"stop"` from `"length"`. What does a caller do
   differently in each case, and what breaks in a chat UI if you always return `"stop"`?
2. `usage` reports token counts for a model nobody is billing for. Name two things other
   than billing that a caller uses those numbers for. Why is it absent while streaming,
   and what does that cost the caller?
3. Your `render_prompt` had to decide what a `system` message means for a model that was
   never fine-tuned with one. Say what you chose and what the alternatives would cost.
   What would it take to support it properly?
4. Streaming returns the same tokens in the same order and changes no metric in Task V
   — total time is unchanged. Name what it does improve, in a term a user would
   recognise, and explain why that matters more at 500 tokens than at 24.
5. The streaming code had to decode the whole completion each step rather than each
   token alone. Explain the failure using byte-level BPE, and connect it to why TP-2's
   `reverse` task was impossible.
6. This server handles one request at a time, in Python, with no batching and no
   authentication. Pick the **one** you would add first for a real deployment and
   justify it against your Task V numbers.
7. You never changed the model in this notebook, only how it is represented and reached.
   Argue for or against: "the hard part of deployment is not the model."

---

-
-
-
-
-
-
-

## Shut it down

The thread is a daemon, so it dies with the kernel — but leaving a socket bound while
you keep working is how the next run greets you with *address already in use*.

In [ ]:
server.should_exit = True
time.sleep(1.0)
print("server stopped")

try:                                  # the Netron viewer, if you started one
    import netron
    netron.stop()
except Exception:
    pass

print("\nartifacts in ./onnx:")
for f in sorted(os.listdir(ONNX_DIR)):
    print(f"  {f:<24} {os.path.getsize(os.path.join(ONNX_DIR, f)) / 1e6:>7.1f} MB")

# Conclusions

-
-
-

# Congratulations! 🎉

Across three notebooks you pretrained a GPT, made it sparse with a Mixture of Experts,
taught it to follow instructions with full fine-tuning and LoRA, exported it to a
portable graph, quantized it, and served it behind the same API the rest of the industry
speaks.

Pretraining, alignment, and deployment — the whole pipeline, at a scale you can read end
to end.